# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

First, filtering out header strings and converting keys to int in both datasets. Then, caching the patent data for later when the joined patent id's are rejoined with the rest of the dataset and taking a 5% sample to save on compute time.

In [24]:
# Keeps the entire original comma-separated line string mapped to its Patent ID
full_raw_patents = rddPatents \
    .filter(lambda line: "PATENT" not in line and len(line.strip()) > 0) \
    .map(lambda line: (int(line.split(",")[0].replace('"', '')), line.strip()))
full_raw_patents.cache()

# 1. Parse Citations: Use 'in' keyword to safely skip quoted headers
parsed_citations = rddCitations \
    .filter(lambda line: "CITING" not in line and len(line.strip()) > 0) \
    .map(lambda line: [int(x.strip().replace('"', '')) for x in line.split(",")])

# Force a small sample for fast local execution testing
# Change back to full dataset only for the final submission run!
parsed_citations = parsed_citations.sample(False, 0.05, seed=42)

# 2. Parse Patents: Clean quotes before parsing elements
parsed_patents = rddPatents \
    .filter(lambda line: "PATENT" not in line and len(line.strip()) > 0) \
    .map(lambda line: [x.strip().replace('"', '') for x in line.split(",")]) \
    .map(lambda cols: (int(cols[0]), cols[5])) \
    .filter(lambda x: x[1] != "" and x[1].lower() != "null")

# Cache to optimize memory processing during the upcoming joins
parsed_patents.cache()

PythonRDD[135] at RDD at PythonRDD.scala:53

We then conduct the multi-join, saving a cache between join steps. The second join rearranges "cited" so that it is the key with each cited state.

In [25]:
# --- JOIN 1: Match Citing Patent State ---
citations_by_citing = parsed_citations.map(lambda x: (x[0], x[1]))
joined_citing = citations_by_citing.join(parsed_patents)

# Cache the output of the first massive join operation
joined_citing.cache()

# --- JOIN 2: Match Cited Patent State ---
# Rearrange so the 'cited' ID becomes the key: (cited_id, (citing_id, citing_state))
# x[0] is citing_id, x[1][0] is cited_id, x[1][1] is citing_state
citations_by_cited = joined_citing.map(lambda x: (x[1][0], (x[0], x[1][1])))

final_joined = citations_by_cited.join(parsed_patents)

Finally, taking the augmented counts and appending to the originally cached raw patent data.

In [26]:
# leftOuterJoin returns a tuple: (Patent_ID, (full_raw_line_string, count_or_None))
augmented_rdd = full_raw_patents.leftOuterJoin(citation_counts) \
    .map(lambda x: f"{x[1][0]},{x[1][1] if x[1][1] is not None else 0}")

# The lambda splits by commas and reads the very last item [-1] (our count)
top_10_augmented = augmented_rdd.top(10, key=lambda line: int(line.split(",")[-1]))

print("Full Augmented Data Lines (Top 10):")
print("-" * 60)
for line in top_10_augmented:
    print(line)

Full Augmented Data Lines (Top 10):
------------------------------------------------------------
5959466,1999,14515,1997,"US","CA",5310,2,,326,4,46,159,0,1,,0.6186,,4.8868,0.0455,0.044,,,125
5983822,1999,14564,1998,"US","TX",569900,2,,114,5,55,200,0,0.995,,0.7201,,12.45,0,0,,,103
6008204,1999,14606,1998,"US","CA",749584,2,,514,3,31,121,0,1,,0.7415,,5,0.0085,0.0083,,,100
5952345,1999,14501,1997,"US","CA",749584,2,,514,3,31,118,0,1,,0.7442,,5.1102,0,0,,,98
5958954,1999,14515,1997,"US","CA",749584,2,,514,3,31,116,0,1,,0.7397,,5.181,0,0,,,96
5998655,1999,14585,1998,"US","CA",,1,,560,1,14,114,0,1,,0.7387,,5.1667,,,,,96
5936426,1999,14466,1997,"US","CA",5310,2,,326,4,46,178,0,1,,0.58,,11.2303,0.0765,0.073,,,94
5739256,1998,13983,1995,"US","CA",70060,2,15,528,1,15,453,0,1,,0.8232,,15.1104,0.1124,0.1082,,,90
5978329,1999,14550,1995,"US","CA",148925,2,,369,2,24,145,0,1,,0.5449,,12.9241,0.4196,0.4138,,,90
5980517,1999,14557,1998,"US","CA",733846,2,,606,3,32,241,0,1,,0.7394,,8.3776,0,0,,,90


Executing the same operations on the entire dataset.

In [27]:
# Keep the entire original comma-separated line string mapped to its Patent ID
full_raw_patents = rddPatents \
    .filter(lambda line: "PATENT" not in line and len(line.strip()) > 0) \
    .map(lambda line: (int(line.split(",")[0].replace('"', '')), line.strip()))
full_raw_patents.cache()

# 1. Parse Citations: Use 'in' keyword to safely skip quoted headers
# NOTE: The .sample() method is removed here to process the full dataset!
parsed_citations = rddCitations \
    .filter(lambda line: "CITING" not in line and len(line.strip()) > 0) \
    .map(lambda line: [int(x.strip().replace('"', '')) for x in line.split(",")])

# 2. Parse Patents: Clean quotes before parsing elements
parsed_patents = rddPatents \
    .filter(lambda line: "PATENT" not in line and len(line.strip()) > 0) \
    .map(lambda line: [x.strip().replace('"', '') for x in line.split(",")]) \
    .map(lambda cols: (int(cols[0]), cols[5])) \
    .filter(lambda x: x[1] != "" and x[1].lower() != "null")

# Cache to optimize memory processing during the upcoming joins
parsed_patents.cache()

# --- JOIN 1: Match Citing Patent State ---
citations_by_citing = parsed_citations.map(lambda x: (x[0], x[1]))
joined_citing = citations_by_citing.join(parsed_patents)

# Cache the output of the first massive join operation
joined_citing.cache()

# --- JOIN 2: Match Cited Patent State ---
# Rearrange so the 'cited' ID becomes the key: (cited_id, (citing_id, citing_state))
# x[0] is citing_id, x[1][0] is cited_id, x[1][1] is citing_state
citations_by_cited = joined_citing.map(lambda x: (x[1][0], (x[0], x[1][1])))

final_joined = citations_by_cited.join(parsed_patents)
# final_joined structure is: (cited_id, ((citing_id, citing_state), cited_state))

# Cache the final double-join to break the lazy execution graph
final_joined.cache()

# Filter down to matches where Citing State matches Cited State
# x[1][0][1] is citing_state, x[1][1] is cited_state
same_state_matches = final_joined.filter(lambda x: x[1][0][1] == x[1][1])

# Aggregate the counts: Map to (citing_id, 1) and sum them up
# x[1][0][0] targets the citing_id
citation_counts = same_state_matches \
    .map(lambda x: (x[1][0][0], 1)) \
    .reduceByKey(lambda a, b: a + b)

# Left Outer Join returns a tuple: (Patent_ID, (full_raw_line_string, count_or_None))
augmented_rdd = full_raw_patents.leftOuterJoin(citation_counts) \
    .map(lambda x: f"{x[1][0]},{x[1][1] if x[1][1] is not None else 0}")

# Extract the top 10 rows based on the newly appended comma-separated count value
top_10_augmented = augmented_rdd.top(10, key=lambda line: int(line.split(",")[-1]))

# Print the final result
print("Full Augmented Data Lines (Top 10):")
print("-" * 60)
for line in top_10_augmented:
    print(line)


Full Augmented Data Lines (Top 10):
------------------------------------------------------------
5959466,1999,14515,1997,"US","CA",5310,2,,326,4,46,159,0,1,,0.6186,,4.8868,0.0455,0.044,,,125
5983822,1999,14564,1998,"US","TX",569900,2,,114,5,55,200,0,0.995,,0.7201,,12.45,0,0,,,103
6008204,1999,14606,1998,"US","CA",749584,2,,514,3,31,121,0,1,,0.7415,,5,0.0085,0.0083,,,100
5952345,1999,14501,1997,"US","CA",749584,2,,514,3,31,118,0,1,,0.7442,,5.1102,0,0,,,98
5958954,1999,14515,1997,"US","CA",749584,2,,514,3,31,116,0,1,,0.7397,,5.181,0,0,,,96
5998655,1999,14585,1998,"US","CA",,1,,560,1,14,114,0,1,,0.7387,,5.1667,,,,,96
5936426,1999,14466,1997,"US","CA",5310,2,,326,4,46,178,0,1,,0.58,,11.2303,0.0765,0.073,,,94
5739256,1998,13983,1995,"US","CA",70060,2,15,528,1,15,453,0,1,,0.8232,,15.1104,0.1124,0.1082,,,90
5978329,1999,14550,1995,"US","CA",148925,2,,369,2,24,145,0,1,,0.5449,,12.9241,0.4196,0.4138,,,90
5980517,1999,14557,1998,"US","CA",733846,2,,606,3,32,241,0,1,,0.7394,,8.3776,0,0,,,90
